# Do-as-I-Do · Retargeting — RunPod

Retargets a reconstructed hand-object demo (the output of the
[`reconstruction/`](../reconstruction/README.md) pipeline) onto a **robot hand** on a RunPod
**A100** pod, from a Jupyter notebook. It consumes the reconstruction output directory directly:
MANO hand tracks + object mesh + per-frame object poses -> gravity-align -> convex decomposition ->
MuJoCo scene -> IK -> sampling-based MPC physics optimization (MuJoCo Warp) -> a robot-hand trajectory.

### RunPod prerequisites (do these in the RunPod dashboard / pod first)
1. **Pod template.** Launch an **NVIDIA** pod (A100 80 GB recommended; >= 24 GB works for shorter
   clips) with a CUDA 12.x PyTorch template. The physics-optimization stage runs sampling-based
   MPC on the GPU via MuJoCo Warp.
2. **Network volume (recommended).** Attach a network volume at `/workspace` so the cloned repo and
   your results persist across pod restarts.
3. **Upload your reconstruction output** onto the pod (RunPod file browser / `scp`). This is the
   `result_data/workspace/` folder from the reconstruction run — it must contain, side-by-side:
   `config.json`, `gravity.json`, `obj_tracking_out/<object>/combined_visualization/layout_camera_frame_optimized.json`,
   `video_segmentation/masks/frame_*_masks/<object>/<object>.obj`, and `<task>/all_hand_meshes.npz`.
   Place its **contents** at `/workspace/raw/` (the default `DATA_DIR` in section 0); the section 6
   resolver will also auto-find it if you drop the folder anywhere under `/workspace`.

Unlike `reconstruction/`, retargeting needs **no HuggingFace auth, no MANO download, no model
weights, and no git submodules** — everything is a single `pip install -e .`.

### What this notebook does
Installs Miniconda, clones the repo, builds the **single `retargeting` conda env**, locates your
reconstruction output, and runs `launch.py --task <task> --raw-dir <raw_dir>` end-to-end.

Run cells top-to-bottom. **[setup]** cells run once per pod; **[run]** cells are per-demo.

> The result is a robot-hand joint trajectory `trajectory_mjwp.npz` written under
> `outputs/<robot>/<hand>/<task>/0/` — the handoff artifact for the [`deployment/`](../deployment/README.md)
> stage.


## 0 · Configuration  [run]

All paths are local filesystem paths on the pod. `TASK` is the video/object name (must match the
object in the reconstruction `config.json`). `DATA_DIR` is where you placed the reconstruction
output; section 6 validates it and resolves the absolute `RAW_DIR`.


In [ ]:
import os

# --- Task / robot (from your reconstruction run) ---
TASK = "pipette"                  # video name == object subfolder name (e.g. "whisking", "pipette")
ROBOT = "sharpa"                  # target robot hand
HAND = "right"                    # "left" | "right" | "bimanual" (also read from config.json)

# --- Where you placed the reconstruction output on the pod ---
# Put the CONTENTS of result_data/workspace/ here (config.json, gravity.json,
# obj_tracking_out/, video_segmentation/, <TASK>/all_hand_meshes.npz). The section 6 resolver also
# auto-searches /workspace if you drop it elsewhere.
DATA_DIR = "/workspace/raw"

# --- Where to clone the repo (network volume = persistence across restarts) ---
REPO_DIR = "/workspace/do-as-i-do"

# --- Run options ---
HEADLESS = True                   # no viser viewer during optimization (RunPod-safe default)
MAX_SIM_STEPS = 0                 # 0 = full trajectory; set N to bound the optimization length

for k, v in {"TASK": TASK, "ROBOT": ROBOT, "HAND": HAND, "DATA_DIR": DATA_DIR,
             "REPO_DIR": REPO_DIR, "HEADLESS": str(HEADLESS),
             "MAX_SIM_STEPS": str(MAX_SIM_STEPS)}.items():
    os.environ[k] = v
print("TASK     =", TASK)
print("ROBOT    =", ROBOT)
print("DATA_DIR =", DATA_DIR)
print("REPO_DIR =", REPO_DIR)


## 1 · GPU, disk & system dependencies  [setup]

Checks the GPU + `nvcc` and installs the system packages several stages need once: `wget`, `zip`
(download cell), and the **EGL/OpenGL libs** MuJoCo's headless renderer uses (`libegl1 libgl1
libgles2`). Doing this up front avoids mid-run failures.


In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
VRAM_MB=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1 | tr -d ' ')
[ "${VRAM_MB:-0}" -lt 24000 ] && echo "!! < 24 GB VRAM may be tight for the 1024-sample MPC (have ${VRAM_MB} MB)"
echo "--- nvcc ---"
nvcc --version 2>/dev/null || ls /usr/local/cuda/bin/nvcc 2>/dev/null || echo "nvcc MISSING - use a CUDA dev image"
echo "--- system deps ---"
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq && apt-get install -y -qq wget zip libegl1 libgl1 libgles2
df -h /workspace 2>/dev/null || df -h /


## 2 · Install Miniconda  [setup]

No-op if the pod image already has conda at `/opt/conda`; otherwise installs Miniconda. Every later
`%%bash` cell re-sources it (shell state does not persist between cells).


In [ ]:
%%bash
set -e
if [ -x /opt/conda/bin/conda ]; then echo "conda already at /opt/conda"; else
  cd /tmp && wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O m.sh
  bash m.sh -bfp /opt/conda && rm m.sh
fi
source /opt/conda/etc/profile.d/conda.sh
conda --version
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true


## 3 · Clone the repo  [setup]

Cloned **without** `--recurse-submodules`: every submodule in this repo lives under
`reconstruction/modules/`, and retargeting depends on none of them (its robot assets are vendored
in-tree under `retargeting/retargeting/assets/`). Skipping the submodules keeps the clone small and
fast. `GIT_LFS_SKIP_SMUDGE=1` is harmless here (no LFS objects needed).


In [ ]:
%%bash
set -e
if [ -d "$REPO_DIR/.git" ]; then echo "repo already at $REPO_DIR"; else
  mkdir -p "$(dirname "$REPO_DIR")"; cd "$(dirname "$REPO_DIR")"
  GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/malik-group/do-as-i-do.git "$(basename "$REPO_DIR")"
fi
cd "$REPO_DIR"
git log --oneline -1
echo "retargeting present:"; ls retargeting/launch.py retargeting/pyproject.toml


## 4 · Create the `retargeting` conda env  [setup]

Unlike `reconstruction/` (four envs), retargeting runs in a **single conda env** installed as a pip
package (`pip install -e .`). `pyproject.toml` pins the known-good `mujoco-warp` commit, which pulls
in matching `mujoco` + `warp-lang` (~1.10).

**torch must match the pod's GPU driver.** The default `pip install torch` wheel is now a **CUDA 13**
build; on a RunPod CUDA 12.x pod its runtime can't init (-> *"NVIDIA driver too old (found version
12080)"*). So we install a torch built for the pod's driver CUDA version (e.g. `cu128` for a 12.8
driver) **before** `pip install -e .`; the editable install then sees `torch` already satisfied and
won't pull the cu13 wheel. `warp-lang`/`mujoco-warp` are torch-version-agnostic. `CUDA_HOME` points
warp at the pod's CUDA toolkit so its GPU kernels compile at runtime.


In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/retargeting"
if ! conda env list | grep -q '^retargeting '; then
  conda create -y -n retargeting python=3.12
fi
conda activate retargeting
pip install -q --upgrade pip

# Pick a torch wheel index that matches the pod driver (default PyPI torch is a cu13 build that
# fails on CUDA 12.x pods with "NVIDIA driver too old"). nvidia-smi prints the max CUDA runtime
# the driver supports, e.g. "CUDA Version: 12.8".
DRV_CUDA=$(nvidia-smi 2>/dev/null | grep -oE 'CUDA Version: [0-9.]+' | head -1 | awk '{print $3}')
echo "driver CUDA Version: ${DRV_CUDA:-unknown}"
case "$DRV_CUDA" in
  12.8*) TORCH_IDX=cu128 ;;
  12.6*) TORCH_IDX=cu126 ;;
  12.4*) TORCH_IDX=cu124 ;;
  12.1*|12.2*|12.3*) TORCH_IDX=cu121 ;;
  *) echo "No CUDA 12.x torch index for '$DRV_CUDA'; defaulting to cu128"; TORCH_IDX=cu128 ;;
esac
echo "installing torch from https://download.pytorch.org/whl/$TORCH_IDX"
pip install -q "torch" --index-url "https://download.pytorch.org/whl/$TORCH_IDX"

export CUDA_HOME="${CUDA_HOME:-/usr/local/cuda}"
pip install -e .
python -c "import torch, mujoco, warp; print('torch', torch.__version__, 'cuda_avail', torch.cuda.is_available()); print('mujoco', mujoco.__version__); print('warp', warp.__version__)"


## 5 · Setup sanity check  [setup]


In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda env list
conda activate retargeting
echo "=== retargeting stack ==="
python - <<'PY'
import torch, mujoco, warp, mink, coacd, trimesh, viser
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu", torch.cuda.is_available())
print("mujoco", mujoco.__version__, "| warp", warp.__version__)
print("mink", getattr(mink, "__version__", "?"), "| coacd", getattr(coacd, "__version__", "?"))
print("trimesh", trimesh.__version__, "| viser", viser.__version__)
if not torch.cuda.is_available():
    raise SystemExit(
        "!! torch.cuda is unavailable - the installed torch build doesn't match the GPU driver. "
        "Re-run section 4 (it now installs a driver-matched cu12x torch)."
    )
PY
echo "=== robot assets present ==="
ls "$REPO_DIR/retargeting/retargeting/assets/robots/$ROBOT/" 2>/dev/null \
  && echo "OK: $ROBOT assets found" || echo "!! missing $ROBOT assets"


---

# Run phase (per-demo)

Re-run from here whenever you change `TASK` / `DATA_DIR`.


## 6 · Locate & validate the reconstruction output  [run]

`process_dataset.py` expects, relative to `raw_dir`: `config.json` (object name + anchor hand),
`gravity.json` (camera->world up vector), `<task>/all_hand_meshes.npz` (HaWoR hand tracks),
`obj_tracking_out/<object>/combined_visualization/layout_camera_frame_optimized.json` (per-frame
object poses), and `video_segmentation/masks/frame_*_masks/<object>/<object>.obj` (object mesh).

The resolver searches `DATA_DIR` (and a few `/workspace` candidates) for the directory containing
`config.json` + `gravity.json` + `obj_tracking_out`, then validates all five artifacts.


In [ ]:
import glob, json, os, sys

TASK = os.environ["TASK"]

SKIP_DIRS = {".git", "do-as-i-do", "__pycache__", ".ipynb_checkpoints", "node_modules"}

def looks_like_raw_dir(d):
    return all(os.path.exists(os.path.join(d, x)) for x in ("config.json", "gravity.json", "obj_tracking_out"))

def find_raw_dir(root, maxdepth=3):
    root = os.path.abspath(root)
    if not os.path.isdir(root):
        return None
    if looks_like_raw_dir(root):
        return root
    for dp in range(1, maxdepth + 1):
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
            depth = os.path.relpath(dirpath, root).count(os.sep)
            if depth >= dp:
                dirnames[:] = []
                continue
            if depth == dp - 1:
                for d in dirnames:
                    cand = os.path.join(dirpath, d)
                    if looks_like_raw_dir(cand):
                        return cand
    return None

raw_dir = None
for cand_root in [os.environ.get("DATA_DIR", "/workspace/raw"),
                  "/workspace/result_data", "/workspace/workspace", "/workspace"]:
    raw_dir = find_raw_dir(cand_root)
    if raw_dir:
        print(f"raw_dir resolved under {cand_root} -> {raw_dir}")
        break

if not raw_dir:
    print("!! Could not find a reconstruction output dir (needs config.json + gravity.json + "
          "obj_tracking_out co-located). Upload result_data/workspace/ and set DATA_DIR in section 0.")
    sys.exit(1)

# --- Validate the 5 required artifacts ---
cfg = json.load(open(os.path.join(raw_dir, "config.json")))
object_name = cfg["object_names"][0]
anchor = cfg.get("anchor_hand", os.environ.get("HAND", "right"))
if object_name != TASK:
    print(f"!! note: config.json object_names[0]={object_name!r} != TASK={TASK!r}; using {object_name!r}.")
    TASK = object_name

checks = {
    "config.json": os.path.join(raw_dir, "config.json"),
    "gravity.json": os.path.join(raw_dir, "gravity.json"),
    "hand_meshes.npz": os.path.join(raw_dir, TASK, "all_hand_meshes.npz"),
    "layout_optimized.json": os.path.join(
        raw_dir, "obj_tracking_out", object_name, "combined_visualization",
        "layout_camera_frame_optimized.json"),
}
mesh_hits = sorted(glob.glob(os.path.join(
    raw_dir, "video_segmentation", "masks", "frame_*_masks", object_name, f"{object_name}.obj")))
checks[f"{object_name}.obj"] = mesh_hits[0] if mesh_hits else "<missing>"

missing = [k for k, p in checks.items() if not os.path.exists(p)]
for k, p in checks.items():
    print(("  OK " if os.path.exists(p) else "MISS ") + f"{k:24s} {p}")
if missing:
    print("\n!! Missing artifacts:", missing)
    sys.exit(1)

os.environ["RAW_DIR"] = raw_dir
os.environ["OBJECT"] = object_name
os.environ["HAND"] = anchor
print(f"\nRAW_DIR = {raw_dir}")
print(f"TASK    = {TASK}  OBJECT = {object_name}  HAND = {anchor}")
print("All required reconstruction artifacts present.")


## 7 · Run the retargeting pipeline  [run]

Runs `python launch.py --task <TASK> --raw-dir <RAW_DIR> --robot-type <ROBOT>` end-to-end (the 5
stages: dataset processing -> convex decomposition -> MJCF scene -> IK -> MuJoCo Warp physics MPC).

`HEADLESS` adds `--no-show-viewer --no-wait-on-finish` (no blocking viser server, so the cell
returns when done). Use section 9 to replay afterward. `MPLBACKEND=Agg` keeps any matplotlib import
headless; `CUDA_HOME` lets warp compile its GPU kernels.


In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate retargeting
cd "$REPO_DIR/retargeting"
export CUDA_HOME="${CUDA_HOME:-/usr/local/cuda}"
export MPLBACKEND=Agg

ARGS=(python launch.py
     --task "$TASK" --raw-dir "$RAW_DIR"
     --robot-type "$ROBOT" --hand-type "$HAND"
     --force)
[ "$HEADLESS" = "True" ]  && ARGS+=(--no-show-viewer --no-wait-on-finish)
[ "$MAX_SIM_STEPS" != "0" ] && ARGS+=(--max-sim-steps "$MAX_SIM_STEPS")

echo "=== ${ARGS[*]} ==="
"${ARGS[@]}"


## 8 · Done — where the outputs live  [run]

Written under `retargeting/outputs/<robot>/<hand>/<task>/0/`. The handoff artifact for the
[`deployment/`](../deployment/README.md) stage is **`trajectory_mjwp.npz`** (the optimized
robot-hand + object trajectory, with per-step tracking-error metrics).


In [ ]:
%%bash
cd "$REPO_DIR/retargeting"
echo "=== run directories ==="
find outputs -maxdepth 4 -type d 2>/dev/null | sort
echo
MJWP="$(find outputs -name trajectory_mjwp.npz -path "*$TASK*" -print -quit 2>/dev/null)"
if [ -n "$MJWP" ]; then
  RUN_DIR="$(dirname "$MJWP")"
  echo "=== run dir: $RUN_DIR ==="
  ls -lh "$RUN_DIR"/
  echo "OK: retargeting finished -> $MJWP"
else
  echo "!! trajectory_mjwp.npz not found under outputs/ for task '$TASK' - check the section 7 log."
fi


---

## 9 · Replay the retargeted trajectory (viser)  [run]

`replay_viser.py` plays a finished run (`scene.xml` + `trajectory_mjwp.npz`) in an interactive
[viser](https://github.com/nerfstudio-project/viser) viewer **without re-running the optimization**.
It **blocks**; reach it from your laptop via SSH port forwarding (run on your **local** machine):
```
ssh -N -L 8081:localhost:8081 root@<pod-ip> -p <port> -i <key>
```
then open `http://localhost:8081` and use the Frame slider / Play button. Stop the cell (square)
when done. (Skip this cell if you ran section 7 with the viewer already enabled.)


In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
conda activate retargeting
cd "$REPO_DIR/retargeting"
RUN_DIR="outputs/$ROBOT/$HAND/$TASK/0"
[ -f "$RUN_DIR/trajectory_mjwp.npz" ] || RUN_DIR="$(dirname "$(find outputs -name trajectory_mjwp.npz -path "*$TASK*" -print -quit)")"
echo "replaying: $RUN_DIR"
ls -lh "$RUN_DIR/scene.xml" "$RUN_DIR/trajectory_mjwp.npz"
python replay_viser.py --run-dir "$RUN_DIR" --port 8081 </dev/null


## 10 · Zip & download results  [run]

Zips the retargeting `outputs/` (scenes, trajectories, object meshes, configs) — everything needed
to replay locally or feed the [`deployment/`](../deployment/README.md) stage. Download with
`scp -P <port> -i <key> root@<pod>:/workspace/retargeting_<task>_results.zip .`


In [ ]:
%%bash
set -e
OUT="/workspace/retargeting_${TASK}_results.zip"
cd "$REPO_DIR/retargeting"
zip -r -q "$OUT" outputs -x "*/.ipynb_checkpoints/*"
ls -lh "$OUT"
